[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/00_setup_verification.ipynb)

# 🌍 Landslide4Sense — Verificación del Entorno

Este notebook verifica que todas las dependencias, datos y módulos estén correctamente configurados para el procesamiento de imágenes multi-espectrales de 14 canales.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

# ── 1. CONFIGURACIÓN DE RUTAS Y ENTORNO ────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
DATA_ROOT_OVERRIDE = None

if IN_COLAB:
    print("📦 Entorno Google Colab detectado. Iniciando configuración...")
    
    # Actualizar API de Kaggle
    subprocess.run([sys.executable, "-m", "pip", "install", "kaggle", "--upgrade", "-q"], check=True)

    # Gestión de credenciales kaggle.json
    from google.colab import files
    kaggle_path = Path('/root/.kaggle/kaggle.json')
    
    if not kaggle_path.exists():
        print("🔑 Sube tu archivo kaggle.json (descárgalo en kaggle.com -> Settings -> API):")
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            kaggle_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.move('kaggle.json', str(kaggle_path))
            kaggle_path.chmod(0o600)
            print("✓ Credenciales configuradas.")
    else:
        print("✓ Credenciales detectadas en /root/.kaggle")

    # Descarga del dataset con manejo de errores robusto
    print("🛰️ Descargando dataset Landslide4Sense...")
    try:
        result = subprocess.run(
            ["kaggle", "datasets", "download", "-d", "lxrzp/landslide4sense", "-p", "/content/", "--unzip"],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f"❌ Error en la descarga: {result.stderr}")
        else:
            print("✓ Descarga y descompresión completada.")
    except Exception as e:
        print(f"❌ Fallo inesperado: {e}")

    # Localización automática de TrainData
    for _root, _dirs, _ in os.walk('/content'):
        if 'TrainData' in _dirs:
            DATA_ROOT_OVERRIDE = _root
            break
else:
    print('💻 Ejecutando en entorno local.')

In [ ]:
# ── 2. VERIFICACIÓN DE ESTRUCTURA DE DATOS ─────────────────────────────────────
if DATA_ROOT_OVERRIDE:
    DATA_ROOT = Path(DATA_ROOT_OVERRIDE)
else:
    _cwd = Path.cwd()
    _candidates = [_cwd / ".." / "data", _cwd / "data", _cwd.parent]
    DATA_ROOT = next((c.resolve() for c in _candidates if (c / "TrainData" / "img").exists()), None)

if DATA_ROOT is None or not (DATA_ROOT / "TrainData").exists():
    print("⚠️ No se encontró la carpeta 'TrainData'.")
else:
    print(f"📂 DATA_ROOT fijado en: {DATA_ROOT}")
    for part in ["TrainData", "ValidData", "TestData"]:
        img_p = DATA_ROOT / part / "img"
        n = len(list(img_p.glob("*.h5"))) if img_p.exists() else 0
        status = "✓" if n > 0 else "✗"
        print(f"  {status} {part:<10} | {n:>4} archivos .h5 encontrados")

In [ ]:
# ── 3. INSTALACIÓN DE LIBRERÍAS Y TEST VISUAL ──────────────────────────────────
if IN_COLAB:
    print("🛠️ Instalando librerías adicionales...")
    subprocess.run([sys.executable, "-m", "pip", "install", "segmentation-models-pytorch", "timm", "h5py", "-q"], check=True)

import torch, h5py, numpy as np
import matplotlib.pyplot as plt

print(f"\n✅ Verificación Técnica:")
print(f"  - PyTorch: {torch.__version__} | GPU: {torch.cuda.is_available()}")

try:
    img_files = sorted((DATA_ROOT / "TrainData" / "img").glob("*.h5"))
    if img_files:
        with h5py.File(img_files[0], 'r') as f:
            sample = f['img'][:]
        
        print(f"  - Patch detectado: {sample.shape} (14 canales OK)")
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        channels = [0, 7, 9] # Rojo, SAR, DEM
        names = ['Canal 0: Rojo', 'Canal 7: SAR VV', 'Canal 9: DEM']
        for i, (ch, name) in enumerate(zip(channels, names)):
            axes[i].imshow(sample[:,:,ch], cmap='gray' if ch!=9 else 'terrain')
            axes[i].set_title(name)
            axes[i].axis('off')
        plt.tight_layout()
        plt.show()
except Exception as e:
    print(f"⚠️ Error visualizando datos: {e}")